# 04 — Feature Engineering and XGBoost

Continues from notebook 03. Logistic regression, even with seller-history
features, shrinkage, interaction terms, and rare-category grouping, plateaued
around AUC-ROC 0.709 / AUC-PR 0.408 — precision at the cost-optimal threshold
was still only around 0.31. This notebook switches the model to XGBoost on
the exact same engineered feature set, to see whether the model family
itself was the limiting factor rather than the features.

Assumes `x_train`, `x_test`, `y_train`, `y_test`, `num_features_v2`,
`cat_features` from notebook 03 are available (re-run 03 in the same
session, or re-execute its cells here first).


In [ ]:
import xgboost as xgb
import shap
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report


## Switching to XGBoost

Logistic regression with this exact feature set plateaued at AUC-PR ~0.42.
Trees can pick up interaction effects and non-linear thresholds (e.g. "risk
rises sharply only once delay exceeds N days") that logistic regression
can't represent without them being hand-engineered. `scale_pos_weight`
handles class imbalance the XGBoost-native way, equivalent to
`class_weight='balanced'` in sklearn.


In [ ]:
preprocessor_xgb = ColumnTransformer([
    ("num", "passthrough", num_features_v2),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

model_xgb = Pipeline([
    ("prep", preprocessor_xgb),
    ("clf", xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr",
        random_state=42,
    ))
])

model_xgb.fit(x_train[num_features_v2 + cat_features], y_train)
probs_xgb = model_xgb.predict_proba(x_test[num_features_v2 + cat_features])[:, 1]

print("AUC-ROC:", roc_auc_score(y_test, probs_xgb))
print("AUC-PR:", average_precision_score(y_test, probs_xgb))
print(classification_report(y_test, probs_xgb > 0.5))


## Cost-optimal threshold

Same cost function as notebook 01/02: a false positive (flagging a good order) costs ₹50, a false negative (missing a bad order) costs ₹300.

In [ ]:
cost_fp, cost_fn = 50, 300
thresholds = np.arange(0.1, 0.9, 0.01)
best_thresh, best_cost = None, float("inf")
for t in thresholds:
    preds = (probs_xgb > t).astype(int)
    fp = ((preds == 1) & (y_test == 0)).sum()
    fn = ((preds == 0) & (y_test == 1)).sum()
    total_cost = fp * cost_fp + fn * cost_fn
    if total_cost < best_cost:
        best_cost, best_thresh = total_cost, t

print(f"Optimal threshold: {best_thresh}, cost: {best_cost}")
print(classification_report(y_test, probs_xgb > best_thresh))


## Overfitting check

Tree ensembles can memorize training data quietly. Comparing train vs. test AUC-PR:

In [ ]:
train_probs_xgb = model_xgb.predict_proba(x_train[num_features_v2 + cat_features])[:, 1]
print("Train AUC-PR:", average_precision_score(y_train, train_probs_xgb))
print("Test AUC-PR:", average_precision_score(y_test, probs_xgb))


## Result

| | AUC-ROC | AUC-PR | Cost | Precision | Recall |
|---|---|---|---|---|---|
| LR baseline (notebook 01/02) | 0.673 | 0.422 | ₹747,350 | 0.25 | 0.56 |
| **XGBoost + new features** | **0.751** | **0.532** | **₹641,050** | **0.37** | **0.54** |

Train AUC-PR (0.589) vs. test AUC-PR (0.532) — a modest ~0.06 gap, consistent
with reasonable generalization rather than overfitting.

This is a genuine, substantial improvement over the logistic regression
baseline: precision nearly kept pace with recall for the first time, and
cost dropped 14%. Continued in notebook 04, which investigates a false-
positive concentration pattern found during error analysis on this model.
